# Train VIMO speech recognition with your private Google Drive

In VIMO, record a variety of real commands under Settings → Train VIMO → Record, then tap Data → Upload to Drive. Include Tamil, Tanglish and English and vary the order of cow name and quantity. In Colab, choose a free GPU runtime if one is available.

The notebook compares fine-tuned Whisper tiny against its starting model on held-out sentences. The report and any improved candidate stay in your Drive.

In [ ]:
!git clone --depth 1 https://github.com/skwatsonoff/vimo-ranch-management.git
%cd vimo-ranch-management/tools/voice_training
!pip install --quiet -r requirements.txt

## Mount Drive and choose the dataset

Sign in with the Google account that owns the Drive folder. Replace the path with the exact folder containing your WAV and JSON files.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_DATASET = Path('/content/drive/MyDrive/VIMO AI/REPLACE_WITH_DATASET_FOLDER')
assert DRIVE_DATASET.is_dir(), 'Enter the actual dataset folder above.'
LOCAL_DATASET = Path('/content/vimo-training-data')
LOCAL_DATASET.mkdir(exist_ok=True)
pairs = [wav for wav in DRIVE_DATASET.glob('*.wav') if wav.with_suffix('.json').is_file()]
assert pairs, 'Upload paired WAV and JSON recordings from VIMO first.'
for wav in pairs:
    shutil.copy2(wav, LOCAL_DATASET)
    shutil.copy2(wav.with_suffix('.json'), LOCAL_DATASET)
print(f'{len(pairs)} recordings available')

## Check and train

Check that both splits contain recordings. Free Colab runtime availability varies. Keep the notebook open while it runs.

In [ ]:
import json
import subprocess
from datetime import datetime, timezone

work = Path('/content/vimo-training-work')
base = ['python', 'finetune_asr.py', '--data', str(LOCAL_DATASET), '--out', str(work)]
subprocess.run([*base, '--dry-run'], check=True)
split = json.loads((work / 'train_report.json').read_text())
assert split['train'] and split['test'], 'Record more distinct commands first.'
subprocess.run([*base, '--evaluate-base', '--max-minutes', '45'], check=True)

report = json.loads((work / 'train_report.json').read_text())
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
destination = DRIVE_DATASET / 'models' / stamp
destination.mkdir(parents=True, exist_ok=True)
if report.get('pack'):
    pack = Path(report['pack'])
    shutil.copy2(pack, destination)
    report['pack'] = pack.name
(destination / 'train_report.json').write_text(json.dumps(report, indent=2))
print(report)
print('Saved in Drive:', destination)

An improved speech candidate needs an iPhone-compatible export and microphone verification before app use. VIMO can already learn its smaller milk-command model from confirmed corrections on the device.